# Feature Engineering Study - NLP (Classification)

Comparative study of textual representations for sentiment classification
on the **Twitter Entity Sentiment Analysis** dataset (73.768 training, 999 validation,
4 classes). Each representation is evaluated with a fixed model (**LinearSVC**),
isolating the effect of feature engineering.

Central question: *which textual representation is most discriminative?*

In [1]:
import numpy as npimport pandas as pdimport reimport timeimport osimport warningswarnings.filterwarnings('ignore')from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizerfrom sklearn.preprocessing import LabelEncoderfrom sklearn.svm import LinearSVCfrom sklearn.metrics import accuracy_score, f1_scoreimport kagglehubSEED = 42np.random.seed(SEED)print('Imports OK')

Imports OK


## 1. Loading and Preprocessing

In [2]:
path = kagglehub.dataset_download('jp797498e/twitter-entity-sentiment-analysis')train_df = pd.read_csv(os.path.join(path, 'twitter_training.csv'))val_df = pd.read_csv(os.path.join(path, 'twitter_validation.csv'))COL_NAMES = ['id', 'entity', 'sentiment', 'text']if train_df.shape[1] == 4:    train_df.columns = COL_NAMES    val_df.columns = COL_NAMESdef clean_tweet(text):    if not isinstance(text, str): return ''    text = text.lower()    text = re.sub(r'http\S+|www\S+', '', text)    text = re.sub(r'@\w+', '', text)    text = re.sub(r'#(\w+)', r'\1', text)    text = re.sub(r'[^a-z0-9\s!?.,\'\-\"]', '', text)    text = re.sub(r'\s+', ' ', text).strip()    return textfor df in (train_df, val_df):    df['clean_text'] = df['text'].apply(clean_tweet)valid_sentiments = ['Positive', 'Negative', 'Neutral', 'Irrelevant']train_df = train_df[train_df['clean_text'].str.len() > 0 & train_df['sentiment'].isin(valid_sentiments)].copy()val_df = val_df[val_df['clean_text'].str.len() > 0 & val_df['sentiment'].isin(valid_sentiments)].copy()le = LabelEncoder()le.fit(valid_sentiments)y_train = le.transform(train_df['sentiment'])y_val = le.transform(val_df['sentiment'])X_train_text = train_df['clean_text'].valuesX_val_text = val_df['clean_text'].valuesprint(f'Train: {len(X_train_text)}, Val: {len(X_val_text)}')

Train: 73768, Val: 999


## 2. Representations Evaluated

| # | Representation | Description |
|---|---------------|-----------|
| 1 | BoW (CountVectorizer) | Raw term counts |
| 2 | TF-IDF (1-gram) | TF-IDF unigrams |
| 3 | TF-IDF (1-2 gram) | TF-IDF uni + bigrams |
| 4 | TF-IDF (1-3 gram) | TF-IDF uni + bi + trigrams |
| 5 | TF-IDF char (2-5) | TF-IDF char n-grams 2-5 |
| 6 | TF-IDF char (3-5) | TF-IDF char n-grams 3-5 |
| 7 | Hashing trick (2^18) | HashingVectorizer 262k dims |
| 8 | Combined word+char | TF-IDF word (50k) + char (20k) |
| 9 | Sentence-BERT | all-MiniLM-L6-v2 frozen (384-d) |

Fixed model: **LinearSVC (C=1.0)** for all representations.

## 3. Results

In [3]:
# Results of the full run (script _run_fe_nlp.py)

Representation                 Acc      F1-w     Feat       Time
--------------------------------------------------------------------
1. BoW (CountVectorizer)       0.9550   0.9550   26351      125.86
2. TF-IDF (1-gram)             0.9419   0.9419   26351      9.01
3. TF-IDF (1-2 gram)           0.9770   0.9770   70000      18.84
4. TF-IDF (1-3 gram)           0.9780   0.9780   70000      15.40
5. TF-IDF char (2-5)           0.9459   0.9460   70000      69.71
6. TF-IDF char (3-5)           0.9439   0.9440   70000      55.42
7. Hashing trick (2^18)        0.9860   0.9860   262144     8.61
8. Combined word+char          0.9820   0.9820   70000      60.67
9. Sentence-BERT               0.6246   0.6129   384        150.72


## 4. Summary Table (sorted by accuracy)

| Representation | Accuracy | F1-w | Features | Time (s) |
|---------------|----------|------|----------|-----------|
| **Hashing trick (2^18)** | **0,9860** | **0,9860** | 262.144 | 8,61 |
| Combined word+char | 0,9820 | 0,9820 | 70.000 | 60,67 |
| TF-IDF (1-3 gram) | 0,9780 | 0,9780 | 70.000 | 15,40 |
| TF-IDF (1-2 gram) | 0,9770 | 0,9770 | 70.000 | 18,84 |
| BoW (CountVectorizer) | 0,9550 | 0,9550 | 26.351 | 125,86 |
| TF-IDF char (2-5) | 0,9459 | 0,9460 | 70.000 | 69,71 |
| TF-IDF char (3-5) | 0,9439 | 0,9440 | 70.000 | 55,42 |
| TF-IDF (1-gram) | 0,9419 | 0,9419 | 26.351 | 9,01 |
| Sentence-BERT | 0,6246 | 0,6129 | 384 | 150,72 |

## 5. Analysis

### 1. Hashing Trick - The Surprising Winner (0,9860 in 8,6s)

HashingVectorizer with 2^18 = 262.144 dimensions outperformed all the other
representations. The reasons for its success:
- **more dimensions than TF-IDF** (262k vs 70k), reducing hash collisions
- **no IDF precision**: the absence of the IDF computation speeds up the fit (8,61s vs
18,84s for TF-IDF), and the linear SVM compensates via the separation margin
- **no vocabulary**: it does not need to store the dictionary, ideal for streaming

### 2. TF-IDF: N-grams Matter (+3,5 pp)

| Representation | Accuracy | Gain vs 1-gram |
|---------------|----------|-----------------|
| TF-IDF (1-gram) | 0,9419 | - |
| TF-IDF (1-2 gram) | 0,9770 | +3,5 pp |
| TF-IDF (1-3 gram) | 0,9780 | +3,6 pp |

Bigrams capture expressions such as "nao gostei" and "muito bom" that unigrams
lose. Trigrams give a marginal gain (+0,1 pp) but increase the time.

### 3. Char n-grams: Inferior to Word n-grams

Char n-grams (0,9459 for 2-5) stay below word n-grams (0,9770 for 1-2).
Char n-grams are useful for texts with typos and regionalisms, but in this clean
(pre-processed) dataset they add noise without benefit. However, the
**word + char** combination (0,9820) beats both of them separately, showing that
char n-grams bring complementary information.

### 4. BoW vs TF-IDF: Does IDF Hurt Here?

Surprisingly, BoW (CountVectorizer, 0,9550) beats TF-IDF 1-gram (0,9419)
by +1,3 pp. This seems counter-intuitive, but it can be explained by the distribution:
since the dataset contains many repetitions (tweets about the same brands), the
raw frequency of relevant terms ("kill", "murder", "love") is more
discriminative than the IDF weighting. Yet BoW is **14x slower** to train
(125,86s vs 9,01s), because the dense counts require more SVM iterations.

### 5. Sentence-BERT: Frozen Remains Inadequate (0,6246)

Confirms the finding of the NLP-twitter-methods-comparasion notebook: the generic
384-d embedding of all-MiniLM-L6-v2 does not capture affective polarity.
Important: here the model is LinearSVC (not LinearSVC with calibration - the result
is slightly worse than the 0,6036 of the previous notebook because it uses a fixed C=1.0).

### Conclusions

1. **Hashing trick is the best cost-benefit**: 0,9860 in 8,6s, with no need
   to build a vocabulary. Ideal for production pipelines.

2. **TF-IDF with n-grams (1-2) is the recommended default**: 0,9770 in 18,8s,
   optimal balance between accuracy and interpretability ( IDF is analyzable).

3. **Combining word + char n-grams brings a real gain** (+0,5 pp over TF-IDF word
   alone), showing that we capture complementary orthographic patterns.

4. **Char n-grams on their own are inferior to word n-grams**, but useful in
   combination or for noisy texts (typos, slang).

5. **Frozen Sentence-BERT is not fit for sentiment classification** - confirmed
   by the second independent experiment.

6. **Final recommendation**: Use Hashing trick for maximum accuracy in minimum time.
   Use TF-IDF (1-2 gram) when you need interpretability. Combine word + char
   when you want to extract the most of classical performance.